# Этап 7. TabM

**TabM** (ICLR 2025) — параметр-эффективный ансамбль из k MLP-голов от Yandex Research.
Единый протокол эксперимента: препроцессоры обучены только на train;
порог классификатора = argmax F2 на val; метрики сохраняются в `results/`.

**Дисбаланс классов**: взвешенный лосс (`BCEWithLogitsLoss` с `pos_weight = N_neg/N_pos`).

## 1. Импорты и настройки

Подключаются библиотеки, фиксируется seed, выбирается устройство и импортируются общие функции метрик.

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import tabm

sys.path.insert(0, str(Path("../src").resolve()))
from utils import compute_metrics, find_best_threshold_f2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"tabm {tabm.__version__} | torch {torch.__version__} | device: {DEVICE}")

tabm 0.0.3 | torch 2.11.0+cu128 | device: cuda


## 2. Загрузка данных

Загружаются готовые train/val/test-разбиения и список числовых и категориальных признаков после preprocessing.

In [2]:
DATA = Path("../data/processed")

train_df = pd.read_parquet(DATA / "train.parquet")
val_df   = pd.read_parquet(DATA / "val.parquet")
test_df  = pd.read_parquet(DATA / "test.parquet")

with open(DATA / "feature_types.json") as f:
    feature_types = json.load(f)

NUM_COLS = feature_types["numeric"]
CAT_COLS = feature_types["categorical"]
TARGET   = "target"

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print(f"Numeric cols: {len(NUM_COLS)}, Categorical cols: {len(CAT_COLS)}")
print(f"Class balance (train): {train_df[TARGET].mean():.3%}")

Train: 41,982  |  Val: 13,994  |  Test: 13,994
Numeric cols: 12, Categorical cols: 13
Class balance (train): 8.971%


## 3. Подготовка числовых признаков

Стандартизация считается только на train, затем применяется к validation и test без утечки данных.

In [3]:
# Fit standardisation only on train
_num_median = train_df[NUM_COLS].median()
_num_mean   = train_df[NUM_COLS].mean()
_num_std    = train_df[NUM_COLS].std().replace(0, 1.0)


def preprocess_num(df: pd.DataFrame) -> np.ndarray:
    X = df[NUM_COLS].fillna(_num_median)
    return ((X - _num_mean) / _num_std).values.astype(np.float32)


X_num_train = preprocess_num(train_df)
X_num_val   = preprocess_num(val_df)
X_num_test  = preprocess_num(test_df)

print("X_num shapes:", X_num_train.shape, X_num_val.shape, X_num_test.shape)

X_num shapes: (41982, 12) (13994, 12) (13994, 12)


## 4. Кодирование категориальных признаков

Категории переводятся в индексы для one-hot представления внутри TabM, неизвестные значения получают отдельный слот.

In [4]:
# TabM uses one-hot encoding internally via cat_cardinalities.
# Build integer index maps from train; +1 slot for unseen categories in val/test.
_cat_maps: dict[str, dict[str, int]] = {}
cat_cardinalities: list[int] = []

for col in CAT_COLS:
    unique_vals = sorted(train_df[col].astype(object).fillna("NA").astype(str).unique())
    _cat_maps[col] = {v: i for i, v in enumerate(unique_vals)}
    cat_cardinalities.append(len(unique_vals) + 1)  # +1 for unseen


def encode_cat(df: pd.DataFrame) -> np.ndarray:
    out = np.empty((len(df), len(CAT_COLS)), dtype=np.int64)
    for j, col in enumerate(CAT_COLS):
        m = _cat_maps[col]
        unk = len(m)  # unseen → last one-hot slot
        strs = df[col].astype(object).fillna("NA").astype(str).tolist()
        out[:, j] = [m.get(s, unk) for s in strs]
    return out


X_cat_train = encode_cat(train_df)
X_cat_val   = encode_cat(val_df)
X_cat_test  = encode_cat(test_df)

total_ohe_dim = sum(cat_cardinalities)
print(f"X_cat shapes: {X_cat_train.shape}")
print(f"Total one-hot dim: {total_ohe_dim}")
print(f"TabM input dim (num + one-hot): {len(NUM_COLS) + total_ohe_dim}")

X_cat shapes: (41982, 13)
Total one-hot dim: 205
TabM input dim (num + one-hot): 217


## 5. TensorDataset и DataLoader

Массивы преобразуются в тензоры PyTorch, после чего собирается DataLoader для обучения.

In [5]:
y_train = train_df[TARGET].values.astype(np.float32)
y_val   = val_df[TARGET].values.astype(np.float32)
y_test  = test_df[TARGET].values.astype(np.float32)

Tn  = torch.from_numpy(X_num_train)
Tc  = torch.from_numpy(X_cat_train)
Ty  = torch.from_numpy(y_train)

Vn  = torch.from_numpy(X_num_val)
Vc  = torch.from_numpy(X_cat_val)

Ten = torch.from_numpy(X_num_test)
Tec = torch.from_numpy(X_cat_test)

BATCH_SIZE = 512
_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(Tn, Tc, Ty),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=_gen,
)

print(f"Train batches/epoch: {len(train_loader)}")

Train batches/epoch: 82


## 6. Архитектура TabM

Задаётся дисбаланс классов через pos_weight и создаётся модель TabM с выбранными размерностями.

In [6]:
N_NUM = len(NUM_COLS)

# Class imbalance: pos_weight = N_neg / N_pos, fit on train
n_pos = int(Ty.sum().item())
n_neg = len(Ty) - n_pos
POS_WEIGHT = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f"pos_weight = {POS_WEIGHT.item():.2f}  (n_neg={n_neg}, n_pos={n_pos})")


def make_model(d_block: int = 256, dropout: float = 0.1, k: int = 16) -> tabm.TabM:
    """Create TabM via .make() which sets sensible defaults for missing args."""
    return tabm.TabM.make(
        n_num_features=N_NUM,
        cat_cardinalities=cat_cardinalities,
        d_out=1,
        d_block=d_block,
        dropout=dropout,
        k=k,
    ).to(DEVICE)


# Smoke-test
_m = make_model()
total_params = sum(p.numel() for p in _m.parameters())
print(f"Parameters (d_block=256, k=16): {total_params:,}")
del _m

pos_weight = 10.15  (n_neg=38216, n_pos=3766)
Parameters (d_block=256, k=16): 226,976


## 7. Функции обучения и предсказания

Описаны один проход обучения, расчёт вероятностей и цикл обучения с ранней остановкой по F2 на validation.

In [7]:
def train_epoch(model, loader, optimizer, criterion) -> float:
    model.train()
    total_loss = 0.0
    for x_n, x_c, y_b in loader:
        x_n, x_c, y_b = x_n.to(DEVICE), x_c.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        out = model(x_n, x_c)                              # (batch, k, 1)
        k = out.shape[1]
        y_exp = y_b.unsqueeze(1).unsqueeze(2).expand(-1, k, 1)
        loss = criterion(out, y_exp)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_proba(
    model, x_num: torch.Tensor, x_cat: torch.Tensor, bs: int = 1024
) -> np.ndarray:
    model.eval()
    probs = []
    for x_n, x_c in DataLoader(TensorDataset(x_num, x_cat), batch_size=bs):
        out = model(x_n.to(DEVICE), x_c.to(DEVICE))       # (batch, k, 1)
        p = torch.sigmoid(out).mean(dim=1).squeeze(-1)     # (batch,)
        probs.append(p.cpu())
    return torch.cat(probs).numpy()


def run_training(
    d_block: int,
    dropout: float,
    k: int,
    max_epochs: int = 80,
    patience: int = 16,
    verbose: bool = True,
) -> tuple:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    model = make_model(d_block=d_block, dropout=dropout, k=k)
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    best_f2, best_state, no_improve = -1.0, None, 0

    for epoch in range(1, max_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, criterion)
        v_proba = predict_proba(model, Vn, Vc)
        thr = find_best_threshold_f2(y_val, v_proba)
        v_f2 = compute_metrics(y_val, v_proba, thr)["f2"]

        if v_f2 > best_f2:
            best_f2 = v_f2
            best_state = {kk: v.cpu().clone() for kk, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(
                f"  Epoch {epoch:3d}  loss={loss:.4f}"
                f"  val_F2={v_f2:.4f}  best={best_f2:.4f}"
            )

        if no_improve >= patience:
            if verbose:
                print(f"  Early stop epoch {epoch}  best val F2={best_f2:.4f}")
            break

    model.load_state_dict(best_state)
    return model, best_f2

## 8. Подбор гиперпараметров

Несколько конфигураций TabM сравниваются только на validation, test на этом этапе не используется.

In [8]:
# Лёгкий подбор гиперпараметров только по val F2 (без теста)
SEARCH_CONFIGS = [
    {"d_block": 128, "dropout": 0.1, "k": 16},
    {"d_block": 256, "dropout": 0.1, "k": 16},
    {"d_block": 128, "dropout": 0.3, "k": 16},
    {"d_block": 256, "dropout": 0.3, "k": 16},
    {"d_block": 512, "dropout": 0.1, "k": 16},
    {"d_block": 256, "dropout": 0.1, "k": 8},
    {"d_block": 256, "dropout": 0.1, "k": 32},
    {"d_block": 256, "dropout": 0.2, "k": 16},
]

print("=== Подбор гиперпараметров (max 30 эпох, patience=8) ===")
search_results = []
for cfg in SEARCH_CONFIGS:
    print(f"\nКонфиг: {cfg}")
    _, val_f2 = run_training(
        d_block=cfg["d_block"],
        dropout=cfg["dropout"],
        k=cfg["k"],
        max_epochs=30,
        patience=8,
        verbose=False,
    )
    print(f"  → val F2 = {val_f2:.4f}")
    search_results.append((val_f2, cfg))

best_cfg = max(search_results, key=lambda x: x[0])[1]
print(f"\nЛучший конфиг: {best_cfg}")
print(f"Лучший val F2 (search): {max(r[0] for r in search_results):.4f}")

=== Подбор гиперпараметров (max 30 эпох, patience=8) ===

Конфиг: {'d_block': 128, 'dropout': 0.1, 'k': 16}
  → val F2 = 0.3705

Конфиг: {'d_block': 256, 'dropout': 0.1, 'k': 16}
  → val F2 = 0.3690

Конфиг: {'d_block': 128, 'dropout': 0.3, 'k': 16}
  → val F2 = 0.3666

Конфиг: {'d_block': 256, 'dropout': 0.3, 'k': 16}
  → val F2 = 0.3684

Конфиг: {'d_block': 512, 'dropout': 0.1, 'k': 16}
  → val F2 = 0.3687

Конфиг: {'d_block': 256, 'dropout': 0.1, 'k': 8}
  → val F2 = 0.3711

Конфиг: {'d_block': 256, 'dropout': 0.1, 'k': 32}
  → val F2 = 0.3689

Конфиг: {'d_block': 256, 'dropout': 0.2, 'k': 16}
  → val F2 = 0.3693

Лучший конфиг: {'d_block': 256, 'dropout': 0.1, 'k': 8}
Лучший val F2 (search): 0.3711


## 9. Финальное обучение

Лучшая конфигурация обучается с увеличенным лимитом эпох и patience.

In [9]:
print(f"=== Финальное обучение: {best_cfg}, max 100 эпох, patience=16 ===")
final_model, best_val_f2 = run_training(
    d_block=best_cfg["d_block"],
    dropout=best_cfg["dropout"],
    k=best_cfg["k"],
    max_epochs=100,
    patience=16,
    verbose=True,
)
print(f"\nЛучший val F2 (final): {best_val_f2:.4f}")

=== Финальное обучение: {'d_block': 256, 'dropout': 0.1, 'k': 8}, max 100 эпох, patience=16 ===
  Epoch   1  loss=1.2233  val_F2=0.3602  best=0.3602
  Epoch  10  loss=1.1008  val_F2=0.3660  best=0.3682
  Epoch  20  loss=0.9481  val_F2=0.3587  best=0.3682
  Early stop epoch 21  best val F2=0.3682

Лучший val F2 (final): 0.3682


## 10. Оценка и сохранение артефактов

Фиксируется F2-оптимальный порог, считаются метрики на train/val/test и сохраняются JSON-метрики и предсказания.

In [10]:
RESULTS = Path("../results")
RESULTS_METRICS = RESULTS / "metrics"
RESULTS_PREDS = RESULTS / "predictions"
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
RESULTS_PREDS.mkdir(parents=True, exist_ok=True)

train_proba = predict_proba(final_model, Tn, Tc)
val_proba = predict_proba(final_model, Vn, Vc)
test_proba = predict_proba(final_model, Ten, Tec)

threshold_f2 = find_best_threshold_f2(y_val, val_proba)
train_metrics = compute_metrics(y_train, train_proba, threshold_f2)
val_metrics = compute_metrics(y_val, val_proba, threshold_f2)
test_metrics = compute_metrics(y_test, test_proba, threshold_f2)

metrics = {
    "model": "tabm",
    "threshold_f2": float(threshold_f2),
    "train": train_metrics,
    "val": val_metrics,
    "test": test_metrics,
    "best_config": best_cfg,
    "best_val_f2": float(best_val_f2),
}

with open(RESULTS_METRICS / "tabm.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

pd.DataFrame({"y_true": y_val.astype(int), "y_proba": val_proba}).to_csv(
    RESULTS_PREDS / "tabm_val.csv", index=False
)
pd.DataFrame({"y_true": y_test.astype(int), "y_proba": test_proba}).to_csv(
    RESULTS_PREDS / "tabm_test.csv", index=False
)

print(f"F2-threshold: {threshold_f2:.4f}")
print(f"Test ROC-AUC: {test_metrics['roc_auc']:.4f}")
print(f"Test F2: {test_metrics['f2']:.4f}")
test_metrics

F2-threshold: 0.4113
Test ROC-AUC: 0.6574
Test F2: 0.3621


{'roc_auc': 0.6574378832890435,
 'precision': 0.1121434036506722,
 'recall': 0.8175298804780876,
 'f2': 0.362058013974169,
 'tp': 1026,
 'fp': 8123,
 'fn': 229,
 'tn': 4616}

## 11. Проверка целевых ориентиров

Test-метрики сравниваются с минимальными и хорошими ориентирами для итогового отчёта.

In [11]:
TARGETS = {
    "roc_auc":   {"min": 0.64, "good": 0.67},
    "precision": {"min": 0.15, "good": 0.20},
    "recall":    {"min": 0.45, "good": 0.60},
    "f2":        {"min": 0.30, "good": 0.38},
}

print("=== Результаты на test vs целевые метрики ===")
print(f"{'Метрика':12} {'Получено':>9} {'Мин':>6} {'Хорошо':>8}  Статус")
all_ok = True
for metric, tgt in TARGETS.items():
    v = test_metrics[metric]
    if v >= tgt["good"]:
        status = "✓ Хорошо"
    elif v >= tgt["min"]:
        status = "✓ Минимум"
    else:
        status = "✗ Ниже цели"
        all_ok = False
    print(f"{metric:12} {v:9.4f} {tgt['min']:6.2f} {tgt['good']:8.2f}  {status}")

print()
if all_ok:
    print("Все целевые метрики достигнуты ✓")
else:
    print(
        "Примечание: для задачи с дисбалансом ~9% и шумных медицинских данных\n"
        "отклонение от целевых метрик допустимо. Разрыв зафиксирован."
    )

=== Результаты на test vs целевые метрики ===
Метрика       Получено    Мин   Хорошо  Статус
roc_auc         0.6574   0.64     0.67  ✓ Минимум
precision       0.1121   0.15     0.20  ✗ Ниже цели
recall          0.8175   0.45     0.60  ✓ Хорошо
f2              0.3621   0.30     0.38  ✓ Минимум

Примечание: для задачи с дисбалансом ~9% и шумных медицинских данных
отклонение от целевых метрик допустимо. Разрыв зафиксирован.
